In [1]:
from langgraph.graph import StateGraph, START, END 
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
import operator
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.7
)

chat_model = ChatHuggingFace(llm=llm)

response = chat_model.invoke("Who are you?")
print(response.content)

I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [4]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

In [5]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser

parser1 = PydanticOutputParser(pydantic_object=SentimentSchema)
parser2 = PydanticOutputParser(pydantic_object=DiagnosisSchema)

In [7]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["negative", "positive"]
    diagnosis: dict
    response: str

In [8]:
def find_sentiment(state: ReviewState):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You must output ONLY valid JSON. No explanations."),
        ("human", """Classify the sentiment.
        Review: {review}
        {format_instructions}
        """)
    ])
    review = state['review']
    chain = prompt | chat_model | parser1
    result = chain.invoke({
        "review": review,
        "format_instructions": parser1.get_format_instructions()
    })
    return {'sentiment': result.sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank you message in response to this review:
    \n\n "{state['review']} \"\n
    Also, kindly ask user to leave feedback on our website. """

    response = chat_model.invoke(prompt)

    return {'response':response}

def run_diagnosis(state: ReviewState):
    review = state["review"]

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You must output ONLY valid JSON that strictly follows the given schema. Do not include explanations or extra text."
        ),
        (
            "human",
            """Diagnose the following negative review and return its issue_type, urgency, and tone.
        Review:
        {review}
        {format_instructions}
        """
        )
    ])
    chain = prompt | chat_model | parser2
    result = chain.invoke({
        "review": review,
        "format_instructions": parser2.get_format_instructions()
    })

    return {"diagnosis": result.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = chat_model.invoke(prompt)

    return {'response': response}


In [9]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [10]:
initial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(initial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'angry', 'urgency': 'high'},
 'response': AIMessage(content="**Case Resolved: Urgent Bug Issue**\n\nDear [User's Name],\n\nI want to start by apologizing for the frustration caused by the bug issue you've been experiencing. I can imagine how frustrating it must be to encounter problems with [software/product name], especially when you need it to work seamlessly.\n\nI've thoroughly reviewed your case and have worked with our team to identify the root cause of the issue. After careful investigation, we've found the solution to the problem. Our team has applied the necessary fixes, and we're confident that the issue has been resolved.\n\nTo ensure that you're back up and running 